In [ ]:
import os

import numpy as np
import xarray as xr
from bluemath_tk.core.io import load_model
from bluemath_tk.core.operations import get_degrees_from_uv, get_uv_components
from bluemath_tk.datamining.mda import MDA
from bluemath_tk.datamining.pca import PCA
from bluemath_tk.interpolation.gps import ExactGPInterpolation

In [2]:
num_mda_points = 200

In [3]:
mda_models = []
pca_models = []
gp_models = []

for goal in range(1, 15):
    print(f"Processing goal {goal}")
    metamodel_cases_goal = xr.open_dataset(
        f"SnapWave_CASES_METAMODEL/10mDepthAndBuoysPoints_{goal}.nc"
    )
    wavedir_u, wavedir_v = get_uv_components(metamodel_cases_goal.point_wavdir)
    metamodel_cases_goal["point_wavdir_u"] = wavedir_u
    metamodel_cases_goal["point_wavdir_v"] = wavedir_v
    metamodel_cases_goal["stations"] = range(338)
    metamodel_cases_df_for_goal = metamodel_cases_goal[
        ["tp", "wd", "ds", "zs"]
    ].to_dataframe()

    if os.path.exists(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl"):
        mda = load_model(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/mda_{num_mda_points}", exist_ok=True)
        mda = MDA(num_centers=num_mda_points)
        mda.fit(metamodel_cases_df_for_goal, directional_variables=["wd"])
        # mda.plot_selected_centroids();
        mda.save_model(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl")
    mda_models.append(mda)

    if os.path.exists(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl"):
        pca = load_model(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/pca_{num_mda_points}", exist_ok=True)
        pca = PCA(n_components=0.95)
        pcs_ds = pca.fit_transform(
            data=metamodel_cases_goal.isel(case_num=mda.centroid_real_indices),
            vars_to_stack=["point_hm0", "point_tp", "point_wavdir_u", "point_wavdir_v"],
            coords_to_stack=["stations"],
            pca_dim_for_rows="case_num",
        )
        pca.save_model(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl")
    pca_models.append(pca)

    if os.path.exists(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl"):
        gp = load_model(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/gp_{num_mda_points}", exist_ok=True)
        gp = ExactGPInterpolation()
        gp.fit(
            subset_data=mda.centroids[["tp", "wd", "ds", "zs"]],
            subset_directional_variables=["wd"],
            target_data=pca.pcs_df,
        )
        gp.save_model(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl")
    gp_models.append(gp)

Processing goal 1
Processing goal 2
Processing goal 3
Processing goal 4
Processing goal 5
Processing goal 6
Processing goal 7
Processing goal 8
Processing goal 9
Processing goal 10
Processing goal 11
Processing goal 12
Processing goal 13
Processing goal 14


In [4]:
len(mda_models), len(pca_models), len(gp_models)

(14, 14, 14)

In [5]:
reconstructed_months = []

for month, month_str in zip(
    ["2023-01"],
    ["202301"],
):
    # Set list to save the predictions
    reconstruction_for_each_goal = []
    # Load the reanalysis timeseries
    reanalysis_ds = (
        xr.open_zarr("inputs/WHACS/NorthCarolina.zarr")
        # .sel(time=slice("1986-01", "1986-03"))
        .sel(time=month)
        .assign_coords(
            {"goal": (["seapoint"], [14, 13, 12, 10, 8, 7, 6, 5, 4, 3, 2, 1, 11, 9])}
        )
        .sortby("goal")
        .load()
    )

    for partition, variables in {
        "bulk": ["hs", "fp", "dp", "spr"],
        "sea": ["phs0", "ptp0", "pdp0", "pspr0"],
        "swell1": ["phs1", "ptp1", "pdp1", "pspr1"],
        "swell2": ["phs2", "ptp2", "pdp2", "pspr2"],
        "swell3": ["phs3", "ptp3", "pdp3", "pspr3"],
    }.items():
        # Iterate over all the goals
        for goal in range(1, 15):
            # First, set the proper variables
            if partition == "bulk":
                reanalysis_ds_for_goal = reanalysis_ds.isel(seapoint=goal - 1)[
                    variables
                ].rename(
                    {
                        "hs": "hs",
                        "fp": "tp",
                        "dp": "wd",
                        "spr": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
                reanalysis_df_for_goal["tp"] = 1 / reanalysis_df_for_goal["tp"]
            elif partition == "sea":
                reanalysis_ds_for_goal = reanalysis_ds.isel(seapoint=goal - 1)[
                    variables
                ].rename(
                    {
                        "phs0": "hs",
                        "ptp0": "tp",
                        "pdp0": "wd",
                        "pspr0": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
            elif partition == "swell1":
                reanalysis_ds_for_goal = reanalysis_ds.isel(seapoint=goal - 1)[
                    variables
                ].rename(
                    {
                        "phs1": "hs",
                        "ptp1": "tp",
                        "pdp1": "wd",
                        "pspr1": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
            elif partition == "swell2":
                reanalysis_ds_for_goal = reanalysis_ds.isel(seapoint=goal - 1)[
                    variables
                ].rename(
                    {
                        "phs2": "hs",
                        "ptp2": "tp",
                        "pdp2": "wd",
                        "pspr2": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
            elif partition == "swell3":
                reanalysis_ds_for_goal = reanalysis_ds.isel(seapoint=goal - 1)[
                    variables
                ].rename(
                    {
                        "phs3": "hs",
                        "ptp3": "tp",
                        "pdp3": "wd",
                        "pspr3": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()

            # Add water level
            reanalysis_df_for_goal["zs"] = 0.0
            # (
            #     xr.open_zarr("inputs/CODEC/processed/CODEC_netherlands.zarr")
            #     .sel(time=month, goal=goal)
            #     .waterlevel.values
            # )

            # Predict in buoys
            prediction_ds_in_points = pca_models[goal - 1].inverse_transform(
                xr.Dataset(
                    {
                        "PCs": (
                            ("case_num", "n_component"),
                            gp_models[goal - 1]
                            .predict(
                                dataset=reanalysis_df_for_goal.fillna(0),
                            )
                            .values,
                        )
                    },
                    coords={"case_num": reanalysis_df_for_goal.index},
                )
            )
            prediction_ds_in_buoys = xr.Dataset(
                {
                    "hs": (
                        ("time", "stations"),
                        prediction_ds_in_points.point_hm0.values,
                    ),
                    "tp": (
                        ("time", "stations"),
                        prediction_ds_in_points.point_tp.values,
                    ),
                    "dp": (
                        ("time", "stations"),
                        get_degrees_from_uv(
                            prediction_ds_in_points.point_wavdir_u.values,
                            prediction_ds_in_points.point_wavdir_v.values,
                        ),
                    ),
                    "spr": (
                        ("time", "stations"),
                        reanalysis_df_for_goal.ds.repeat(338).values.reshape(744, 338),
                    ),
                },
                coords={
                    "time": reanalysis_df_for_goal.index,
                },
            )
            prediction_ds_in_buoys["hs"] = (
                prediction_ds_in_buoys.hs * reanalysis_ds_for_goal.hs
            )
            prediction_ds_in_buoys = (
                prediction_ds_in_buoys.expand_dims(
                    {"goal": [goal], "partition": [partition]}
                )
                # .isel(
                #     stations=[
                #         -10,
                #         -12,
                #         -30,
                #         -34,
                #         -15,
                #         -14,
                #         -26,
                #         -27,
                #         -13,
                #         -25,
                #         -16,
                #         -24,
                #         -11,
                #         -36,
                #         -35,
                #     ]
                # )
                .load()
                .drop(["seapoint"])
            )

            # Save the prediction in buoys
            reconstruction_for_each_goal.append(prediction_ds_in_buoys)

INFO:ExactGPInterpolation:Preprocessing dataset for prediction
INFO:ExactGPInterpolation:Checking for NaNs in subset data
INFO:ExactGPInterpolation:Preprocessing subset data
INFO:ExactGPInterpolation:Normalizing subset data
INFO:ExactGPInterpolation:Proposed max custom scaler for ds is lower than datapoint
INFO:ExactGPInterpolation:Proposed max custom scaler for wd_u is lower than datapoint
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_v is bigger than datapoint
INFO:ExactGPInterpolation:Subset data preprocessed successfully
INFO:ExactGPInterpolation:Predicting target variable: PC1
INFO:ExactGPInterpolation:Predicting target variable: PC2
INFO:ExactGPInterpolation:Predicting target variable: PC3
INFO:ExactGPInterpolation:Predicting target variable: PC4
INFO:ExactGPInterpolation:Predicting target variable: PC5
INFO:ExactGPInterpolation:Predicting target variable: PC6
INFO:ExactGPInterpolation:Predicting target variable: PC7
INFO:ExactGPInterpolation:Predicting target varia

In [6]:
len(reconstruction_for_each_goal), reconstruction_for_each_goal[0]

(70,
 <xarray.Dataset> Size: 8MB
 Dimensions:    (goal: 1, partition: 1, time: 744, stations: 338)
 Coordinates:
   * goal       (goal) int64 8B 1
   * partition  (partition) object 8B 'bulk'
   * time       (time) datetime64[ns] 6kB 2023-01-01 ... 2023-01-31T23:00:00
 Dimensions without coordinates: stations
 Data variables:
     hs         (goal, partition, time, stations) float64 2MB 3.1e-06 ... 0.00...
     tp         (goal, partition, time, stations) float64 2MB 0.0 0.0 ... 11.8
     dp         (goal, partition, time, stations) float64 2MB 270.0 ... 94.58
     spr        (goal, partition, time, stations) float64 2MB 25.5 25.5 ... 20.1)

In [7]:
datasesito = xr.merge(reconstruction_for_each_goal)
datasesito

<xarray.Dataset> Size: 563MB
Dimensions:    (time: 744, goal: 14, partition: 5, stations: 338)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2023-01-01 ... 2023-01-31T23:00:00
  * goal       (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * partition  (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
Dimensions without coordinates: stations
Data variables:
    hs         (goal, partition, time, stations) float64 141MB 3.1e-06 ... 0....
    tp         (goal, partition, time, stations) float64 141MB 0.0 0.0 ... 4.091
    dp         (goal, partition, time, stations) float64 141MB 270.0 ... 186.6
    spr        (goal, partition, time, stations) float64 141MB 25.5 ... 16.0

In [8]:
datasesito.nbytes * 1e-9

0.563303384

In [46]:
from wavespectra.construct import construct_partition

reconstructed_sites = []
sites_to_reconstruct = range(338)
for site_to_reconstruct in sites_to_reconstruct:
    print(f"Reconstructing {site_to_reconstruct} \r")
    partitions = (
        datasesito.isel(partition=[1, 2, 3, 4], stations=[site_to_reconstruct])
        .stack(part=("goal", "partition"))
        .rename({"stations": "site"})
    )
    partitions["site"] = [site_to_reconstruct]
    dir = np.arange(0, 360, 5)
    freq = np.logspace(np.log10(0.035), np.log10(0.5), 29)
    full_spec = construct_partition(
        freq_name="jonswap",
        freq_kwargs={
            "freq": freq,
            "fp": 1 / partitions.tp,
            "hs": partitions.hs,
        },
        dir_name="cartwright",
        dir_kwargs={
            "dir": dir,
            "dm": partitions.dp,
            "dspr": partitions.spr,
        },
    )
    bulk_by_site = full_spec.sum(dim="part").spec.stats(["hs", "tp", "dm"])
    reconstructed_sites.append(bulk_by_site)

Reconstructing 0 
Reconstructing 1 
Reconstructing 2 
Reconstructing 3 
Reconstructing 4 
Reconstructing 5 
Reconstructing 6 
Reconstructing 7 
Reconstructing 8 
Reconstructing 9 
Reconstructing 10 
Reconstructing 11 
Reconstructing 12 
Reconstructing 13 
Reconstructing 14 
Reconstructing 15 
Reconstructing 16 
Reconstructing 17 
Reconstructing 18 
Reconstructing 19 
Reconstructing 20 
Reconstructing 21 
Reconstructing 22 
Reconstructing 23 
Reconstructing 24 
Reconstructing 25 
Reconstructing 26 
Reconstructing 27 
Reconstructing 28 
Reconstructing 29 
Reconstructing 30 
Reconstructing 31 
Reconstructing 32 
Reconstructing 33 
Reconstructing 34 
Reconstructing 35 
Reconstructing 36 
Reconstructing 37 
Reconstructing 38 
Reconstructing 39 
Reconstructing 40 
Reconstructing 41 
Reconstructing 42 
Reconstructing 43 
Reconstructing 44 
Reconstructing 45 
Reconstructing 46 
Reconstructing 47 
Reconstructing 48 
Reconstructing 49 
Reconstructing 50 
Reconstructing 51 
Reconstructing 52 
Rec

In [47]:
reconstructed_sites[-1]

<xarray.Dataset> Size: 21kB
Dimensions:  (time: 744, site: 1)
Coordinates:
  * time     (time) datetime64[ns] 6kB 2023-01-01 ... 2023-01-31T23:00:00
  * site     (site) int64 8B 337
Data variables:
    hs       (time, site) float64 6kB 1.744 1.816 1.854 ... 0.7683 0.7668 0.7645
    tp       (time, site) float32 3kB dask.array<chunksize=(744, 1), meta=np.ndarray>
    dm       (time, site) float64 6kB 123.9 129.4 134.0 ... 106.2 105.5 106.5
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

In [49]:
xr.concat(reconstructed_sites, dim="site").to_netcdf("outputs/bulk_metamodel.nc")